# Table 1 — Robustness of classical deep hedging (Black-Scholes)

Replicates Table 1 of He, Sutter & Gonon (2025):  
**Robustness of classical deep hedging strategy under different attack methods and magnitudes.**

For each CVaR model (α ∈ {0.5, 0.75, 0.95, 0.99}) and each perturbation radius δ,  
we evaluate the ES_α hedging loss on adversarially perturbed GBM paths using:
- **WPGD** — Wasserstein-2 distributional attack (Frank-Wolfe)
- **WBPGD** — Per-path budget / direction decomposition attack

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
from pathlib import Path

from src.gbm_simulator import GBMParams, GBMPathGenerator
from src.hedging.hedge_network import HedgeNet
from src.hedging.loss import CVaRLoss
from src.hedging.attacker import DHAttacker

RESULTS_DIR = Path('..') / 'results'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on {device}')

In [ ]:
# Parameters (match training in train_buehler_benchmark.py)
S0, K, sigma, mu = 100.0, 100.0, 0.2, 0.0
N, dt = 30, 1 / 365
T = N * dt

ALPHAS     = [0.5, 0.75, 0.95, 0.99]
DELTA_LIST = [0.0, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
RATIO      = 4
N_ITER     = 20

In [ ]:
# Generate clean test paths (seed=20 matches buehler_benchmark.ipynb)
params = GBMParams(S0=S0, mu=mu, sigma=sigma, T=T, N=N, M=100_000)
paths  = GBMPathGenerator(params)(seed=20).to(device)
print(f'Test paths: {paths.shape}  on {paths.device}')

In [ ]:
# Load trained networks + saved z / capital for each alpha
TAGS = {0.5: 'ES05', 0.75: 'ES075', 0.95: 'ES095', 0.99: 'ES099'}

models = {}
for alpha in ALPHAS:
    tag = TAGS[alpha]
    net = HedgeNet(N=N, width=20)
    net.load_state_dict(
        torch.load(RESULTS_DIR / f'buehler_{tag}_network.pt', weights_only=True)
    )
    net = net.to(device).eval()

    log      = torch.load(RESULTS_DIR / f'buehler_{tag}_log.pt', weights_only=False)
    z        = torch.tensor(log['z'],       dtype=torch.float32, device=device)
    capital  = torch.tensor(log['capital'], dtype=torch.float32, device=device)
    models[alpha] = (net, z, capital)
    print(f'  alpha={alpha:4}  z={log["z"]:+.5f}  capital={log["capital"]:.4f}')

In [ ]:
def eval_es(X: torch.Tensor, alpha: float) -> float:
    """Empirical ES_alpha of hedging errors X, evaluated at optimal (empirical) VaR."""
    z_opt = torch.quantile(X, alpha)
    return (torch.relu(X - z_opt).mean() / (1.0 - alpha) + z_opt).item()

In [ ]:
attacker = DHAttacker()
results  = {}   # (alpha, delta) -> {'wpgd': float, 'wbpgd': float}

for alpha in ALPHAS:
    net, z, capital = models[alpha]
    loss_fn = CVaRLoss(K=K, alpha=alpha)
    print(f'\n=== ES_{alpha} ===')

    for delta in DELTA_LIST:
        # WPGD — Wasserstein-2 distributional attack
        _, _, X_wp = attacker.wp_attack_gbm(
            net, paths, loss_fn, capital, z,
            delta=delta, ratio=RATIO, n=N_ITER,
        )

        # WBPGD — per-path budget / direction attack
        _, _, X_bp = attacker.budget_attack_gbm(
            net, paths, loss_fn, capital, z,
            delta=delta, ratio=RATIO, n=N_ITER,
        )

        es_wp = eval_es(X_wp.to(device), alpha)
        es_bp = eval_es(X_bp.to(device), alpha)
        results[(alpha, delta)] = {'wpgd': es_wp, 'wbpgd': es_bp}
        print(f'  delta={delta:.2f}  WPGD={es_wp:.4f}  WBPGD={es_bp:.4f}')

In [ ]:
# Table 1: rows = delta, columns = (alpha, attack_method)
col_idx = pd.MultiIndex.from_tuples(
    [(f'ES_{a}', m) for a in ALPHAS for m in ['WPGD', 'WBPGD']],
    names=['Model', 'Attack'],
)

rows = []
for delta in DELTA_LIST:
    row = []
    for alpha in ALPHAS:
        row += [results[(alpha, delta)]['wpgd'], results[(alpha, delta)]['wbpgd']]
    rows.append(row)

df = pd.DataFrame(
    rows,
    index=pd.Index(DELTA_LIST, name='δ'),
    columns=col_idx,
)

print('Table 1 — ES_α hedging loss under adversarial attacks (GBM / BS model)\n')
print(df.to_string(float_format='{:.4f}'.format))
df.style.format('{:.4f}')